In [33]:
import pandas as pd
import numpy as np

In [34]:
df = pd.read_parquet('./data/yellow_tripdata_2026-02.parquet', columns=["tpep_pickup_datetime", "PULocationID", "passenger_count", "trip_distance"])
df.head()

,tpep_pickup_datetime,PULocationID,passenger_count,trip_distance
0,2026-02-01 00:05:57,107,1.0,0.94
1,2026-02-01 00:35:58,234,1.0,1.93
2,2026-02-01 00:08:41,138,1.0,9.99
3,2026-02-01 00:29:06,209,0.0,1.70
4,2026-02-01 00:53:52,249,0.0,3.70


In [35]:
def time_bucket(hour: int):

    if 0 <= hour <= 3:
        return 'latenight'
    elif 4 <= hour <= 6:
        return 'earlymorning'
    elif 7 <= hour <= 10:
        return 'morning'
    elif 11 <= hour <= 15:
        return 'afternoon'
    elif 16 <= hour <= 20:
        return 'evening'
    else:
        return 'night'

def month_phase(day):
    if day <= 10:
        return "start"
    elif day <= 20:
        return "mid"
    else:
        return "end"

In [36]:
df["pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])

df["hour"] = df["pickup_datetime"].dt.hour
df["date"] = df["pickup_datetime"].dt.date

df["time_bucket"] = df["hour"].apply(time_bucket)

df.drop(columns=["hour", "pickup_datetime", "tpep_pickup_datetime"], inplace=True)

med = df["passenger_count"].median()
df.loc[df["passenger_count"] == 0.0, "passenger_count"] = np.nan
df["passenger_count"] = df["passenger_count"].fillna(med)

med = df["trip_distance"].median()
df.loc[df["trip_distance"] == 0.0, "trip_distance"] = np.nan
df["trip_distance"] = df["trip_distance"].fillna(med)

df.head()

,PULocationID,passenger_count,trip_distance,date,time_bucket
0,107,1.0,0.94,2026-02-01,latenight
1,234,1.0,1.93,2026-02-01,latenight
2,138,1.0,9.99,2026-02-01,latenight
3,209,1.0,1.70,2026-02-01,latenight
4,249,1.0,3.70,2026-02-01,latenight


In [37]:
df['trip_distance'].value_counts()

trip_distance
1.80         145005
0.90          33565
1.00          33238
0.80          33206
1.10          32086
              ...  
96.90             1
118.50            1
190858.87         1
92400.47          1
87726.16          1
Name: count, Length: 4718, dtype: int64

In [38]:
agg_df = df.groupby(
    ["PULocationID", "date", "time_bucket"]
).agg(
    demand=("PULocationID", "size"),
    avg_passengers=("passenger_count", "mean"),
    avg_distance=("trip_distance", "mean")
).reset_index()

agg_df["dayofweek"] = pd.to_datetime(agg_df["date"]).dt.dayofweek

agg_df["loc_time"] = (
    agg_df["PULocationID"].astype(str) + "_" + agg_df["time_bucket"]
)

agg_df.head()

,PULocationID,date,time_bucket,demand,avg_passengers,avg_distance,dayofweek,loc_time
0,1,2026-02-01,afternoon,7,1.857143,1.80,6,1_afternoon
1,1,2026-02-01,evening,6,1.000000,1.80,6,1_evening
2,1,2026-02-01,morning,1,1.000000,1.80,6,1_morning
3,1,2026-02-02,afternoon,4,3.000000,1.80,0,1_afternoon
4,1,2026-02-02,evening,3,1.333333,6.54,0,1_evening


In [39]:
agg_df = agg_df.sort_values(["loc_time", "date"])

# Lag features
agg_df["lag_1"] = agg_df.groupby("loc_time")["demand"].shift(1)
agg_df["lag_2"] = agg_df.groupby("loc_time")["demand"].shift(2)
agg_df["lag_7"] = agg_df.groupby("loc_time")["demand"].shift(7)

# Rolling features
agg_df["rolling_mean_3"] = agg_df.groupby("loc_time")["demand"].transform(
    lambda x: x.shift(1).rolling(3).mean()
)
agg_df["rolling_mean_7"] = agg_df.groupby("loc_time")["demand"].transform(
    lambda x: x.shift(1).rolling(7).mean()
)

agg_df["rolling_std_3"] = agg_df.groupby("loc_time")["demand"].transform(
    lambda x: x.shift(1).rolling(3).std()
)

# Drop NaNs (IMPORTANT)
agg_df = agg_df.dropna()

# 🔥 KEEP base features
# DO NOT drop these
# PULocationID + time_bucket are important
agg_df["is_weekend"] = agg_df["dayofweek"].isin([5,6]).astype(int)

# Encode only time_bucket (small cardinality)
agg_df = pd.get_dummies(agg_df, columns=["time_bucket"], drop_first=True, dtype=int)

# Optional: encode location smartly
freq = agg_df["PULocationID"].value_counts()
agg_df["loc_freq"] = agg_df["PULocationID"].map(freq)

# Drop helper column
agg_df = agg_df.drop("loc_time", axis=1)
agg_df

,PULocationID,date,demand,avg_passengers,avg_distance,dayofweek,lag_1,lag_2,lag_7,rolling_mean_3,rolling_mean_7,rolling_std_3,is_weekend,time_bucket_earlymorning,time_bucket_evening,time_bucket_latenight,time_bucket_morning,time_bucket_night,loc_freq
13084,100,2026-02-08,332,1.274096,2.929729,6,389.0,332.0,389.0,361.666667,350.285714,28.571548,1,0,0,0,0,0,126
13090,100,2026-02-09,324,1.182099,2.719568,0,332.0,389.0,289.0,351.000000,342.142857,32.908965,0,0,0,0,0,0,126
13096,100,2026-02-10,316,1.183544,2.448513,1,324.0,332.0,329.0,348.333333,347.142857,35.444793,0,0,0,0,0,0,126
13102,100,2026-02-11,396,1.166667,2.471111,2,316.0,324.0,360.0,324.000000,345.285714,8.000000,0,0,0,0,0,0,126
13108,100,2026-02-12,401,1.182045,2.485511,3,396.0,316.0,364.0,345.333333,350.428571,44.060564,0,0,0,0,0,0,126
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,9,2026-02-25,3,1.000000,2.650000,2,5.0,2.0,3.0,3.000000,3.000000,1.732051,0,0,0,0,1,0,41
758,9,2026-02-26,2,1.000000,3.720000,3,3.0,5.0,3.0,3.333333,3.000000,1.527525,0,0,0,0,1,0,41
762,9,2026-02-27,1,1.000000,3.060000,4,2.0,3.0,5.0,3.333333,2.857143,1.527525,0,0,0,0,1,0,41
747,9,2026-02-23,1,1.000000,2.470000,0,2.0,1.0,1.0,1.666667,1.714286,0.577350,0,0,0,0,0,1,41


In [40]:
agg_df = agg_df.sort_values("date")

split_date = agg_df["date"].quantile(0.8)

train = agg_df[agg_df["date"] <= split_date]
test  = agg_df[agg_df["date"] > split_date]

In [41]:
X_train = train.drop(columns=["demand", "date"])
y_train = train["demand"]

X_test = test.drop(columns=["demand", "date"])
y_test = test["demand"]

In [42]:
param_dist = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

In [43]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=3)

In [44]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

model = XGBRegressor(random_state=42)

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=10,
    cv=tscv,
    scoring="neg_mean_absolute_error",
    verbose=1,
    n_jobs=-1
)

search.fit(X_train, y_train)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBRegressor(...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.8, 1.0], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [4, 6, ...], 'n_estimators': [100, 200, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies tha

In [45]:
best_model = search.best_estimator_

print(search.best_params_)

{'subsample': 1.0, 'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.1, 'colsample_bytree': 1.0}


In [46]:
y_pred = best_model.predict(X_test)

In [47]:
from sklearn.metrics import mean_absolute_error

print(mean_absolute_error(y_test, y_pred))

15.884578704833984


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mean_demand = y_test.mean()

print(mae / mean_demand)

0.15055003886556173


In [50]:
import joblib

joblib.dump(best_model, "models/demand_model.pkl")

['models/demand_model.pkl']

In [18]:
import glob
import pandas as pd

files = sorted(glob.glob("../windowData/*.parquet"))
print(files)

for i in files:
    df = pd.read_parquet(i, columns=["tpep_pickup_datetime", "PULocationID", "passenger_count", "trip_distance"])
    print(df)

['../windowData\\yellow_tripdata_2024-10.parquet', '../windowData\\yellow_tripdata_2024-11.parquet', '../windowData\\yellow_tripdata_2024-12.parquet']
        tpep_pickup_datetime  PULocationID  passenger_count  trip_distance
0        2024-10-01 00:30:44           162              1.0           3.00
1        2024-10-01 00:12:20            48              1.0           2.20
2        2024-10-01 00:04:46           142              1.0           2.70
3        2024-10-01 00:12:10           233              1.0           3.10
4        2024-10-01 00:30:22           262              1.0           0.00
...                      ...           ...              ...            ...
3833766  2024-10-31 23:49:01           107              NaN           3.49
3833767  2024-10-31 23:35:15           137              NaN           2.40
3833768  2024-10-31 23:30:43           188              NaN          12.28
3833769  2024-10-31 23:00:00           230              NaN           0.56
3833770  2024-10-31 23:1

In [23]:
year, month = map(int, files[-1].split("_")[-1].split(".")[0].split("-"))

if month == 12:
    year += 1
    month = 1
else:
    month += 1

print(f"models/demand_model_{year}-{month:02d}")

models/demand_model_2025-01
